In [40]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "3"

import torch
from human_body_prior.body_model.body_model import BodyModel
import trimesh

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
body_model = BodyModel(bm_fname="./data/SMPLX_NEUTRAL.npz", num_betas=10,  model_type="smplx").to(device).eval()


In [45]:
# 加载pvcp参数
import json
import numpy as np

pvcp_smpl_file="../../output/smplx/pvcp_smplx.json"

with open(pvcp_smpl_file, "r") as f:
    data = json.load(f)

data_i = data['S000_frame_000018.png']
data_i.keys()


dict_keys(['global_orient', 'transl', 'body_pose', 'betas', 'left_hand_pose', 'right_hand_pose', 'expression'])

In [46]:
# ---------- 构建 SMPL-X 输入 ----------
def to_t(x):
    """list/np -> torch (1, D) on device"""
    x = np.asarray(x, dtype=np.float32)
    t = torch.from_numpy(x).to(device)
    if t.ndim == 1:
        t = t.unsqueeze(0)
    return t

# 拼手： (45,) + (45,) -> (90,)
pose_hand = np.concatenate([data_i["left_hand_pose"], data_i["right_hand_pose"]], axis=0)

params = {
    "root_orient": to_t(data_i["global_orient"]),
    "trans":       to_t(data_i["transl"]),
    "pose_body":   to_t(data_i["body_pose"]),
    "betas":       to_t(data_i["betas"]),
    "pose_hand":   to_t(pose_hand),
    "expression":  to_t(data_i["expression"]),
    # JSON里没有的，补零
    "pose_jaw": torch.zeros(1, 3, device=device),
    "pose_eye": torch.zeros(1, 6, device=device),
}

# （可选）打印检查每个输入的 shape
for k, v in params.items():
    print(f"{k:12s} -> {tuple(v.shape)}")

root_orient  -> (1, 3)
trans        -> (1, 3)
pose_body    -> (1, 63)
betas        -> (1, 10)
pose_hand    -> (1, 90)
expression   -> (1, 10)
pose_jaw     -> (1, 3)
pose_eye     -> (1, 6)


In [47]:
# ---------- forward ----------
with torch.no_grad():
    out = body_model(**params)

# out.v: (1, V, 3)
verts = out.v[0].detach().cpu().numpy()
faces = body_model.f  # (F, 3)

mesh = trimesh.Trimesh(vertices=verts, faces=faces, process=False)

# 导出
# mesh.export("pvcp_smplx_frame_000011.obj")
# print("[OK] exported: pvcp_smplx_frame_000011.obj")

# 可视化
mesh.show()